# Loss Ablation Study

**Reference Model:** The 2.5D DecAtt configuration from the main ablation study was used as the reference model. Its results are reported in the Main Ablation Study and are not repeated in this notebook. This experiment focuses on evaluating alternative loss function configurations while keeping the remaining experimental settings unchanged.

Loss Function Ablation Cell

In [ ]:
# Loss Function Cell (Loss Ablation)
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=1.33, eps=1e-6):
        super().__init__()
        self.a, self.b, self.g, self.eps = alpha, beta, gamma, eps
        
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        tp = (p * targets).sum(dim=(2, 3))
        fp = (p * (1 - targets)).sum(dim=(2, 3))
        fn = ((1 - p) * targets).sum(dim=(2, 3))
        t = (tp + self.eps) / (tp + self.a*fp + self.b*fn + self.eps)
        return (1.0 - t).clamp_min(1e-8).pow(self.g).mean()


class SoftDiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
        
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        num = 2 * (p * targets).sum(dim=(2, 3))
        den = (p + targets).sum(dim=(2, 3)) + self.eps
        return (1.0 - (num / den)).mean()


class BoundaryLoss(nn.Module):
    def __init__(self):
        super().__init__()

        kx = torch.tensor(
            [[[-1,0,1],[-2,0,2],[-1,0,1]]],
            dtype=torch.float32
        ).view(1,1,3,3)

        ky = torch.tensor(
            [[[1,2,1],[0,0,0],[-1,-2,-1]]],
            dtype=torch.float32
        ).view(1,1,3,3)

        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)
        
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)

        gx_p = F.conv2d(p, self.kx, padding=1)
        gy_p = F.conv2d(p, self.ky, padding=1)
        gx_t = F.conv2d(targets, self.kx, padding=1)
        gy_t = F.conv2d(targets, self.ky, padding=1)
        
        bp = torch.sqrt(gx_p**2 + gy_p**2 + 1e-6)
        bt = torch.sqrt(gx_t**2 + gy_t**2 + 1e-6)

        return F.l1_loss(bp, bt)


class ComboDiceTverskyLoss(nn.Module):

    def __init__(
        self,
        mode="full",
        w_ft=0.25,
        w_dice=0.25,
        w_bce=0.35,
        w_bd=0.15,
        pos_weight=4.0
    ):
        super().__init__()

        self.mode = mode

        self.ft = FocalTverskyLoss()
        self.dice = SoftDiceLoss()
        self.bd = BoundaryLoss()

        self.weights = [w_ft, w_dice, w_bce, w_bd]

        self.register_buffer(
            "posw",
            torch.tensor(pos_weight)
        )

    def forward(
        self,
        logits,
        targets,
        aux2=None,
        aux3=None,
        attn_mask=None
    ):

        targets_m = (
            targets
            if logits.shape[-2:] == targets.shape[-2:]
            else F.interpolate(
                targets,
                size=logits.shape[-2:],
                mode='nearest'
            )
        )

        # =============================================================
        # 1. Simple Dice + BCE
        # =============================================================
        if self.mode == "simple":

            loss = (
                0.5 * self.dice(logits, targets_m)
                +
                0.5 * F.binary_cross_entropy_with_logits(
                    logits,
                    targets_m
                )
            )

            return loss

        # =============================================================
        # 2. Hybrid loss
        # =============================================================
        loss = (
            self.weights[0] * self.ft(logits, targets_m)
            +
            self.weights[1] * self.dice(logits, targets_m)
            +
            self.weights[2] * F.binary_cross_entropy_with_logits(
                logits,
                targets_m,
                pos_weight=self.posw
            )
            +
            self.weights[3] * self.bd(logits, targets_m)
        )

        # =============================================================
        # 3. Hybrid + Auxiliary
        # =============================================================
        if self.mode in ["aux", "full"]:

            if aux2 is not None:
                t2 = F.interpolate(
                    targets,
                    size=aux2.shape[-2:],
                    mode='nearest'
                )

                loss += 0.15 * F.binary_cross_entropy_with_logits(
                    aux2,
                    t2,
                    pos_weight=self.posw
                )

            if aux3 is not None:
                t3 = F.interpolate(
                    targets,
                    size=aux3.shape[-2:],
                    mode='nearest'
                )

                loss += 0.10 * F.binary_cross_entropy_with_logits(
                    aux3,
                    t3,
                    pos_weight=self.posw
                )

        # =============================================================
        # 4. Hybrid + Auxiliary + Attention guidance
        # =============================================================
        if self.mode == "full":

            if attn_mask is not None:

                t_attn = F.interpolate(
                    targets,
                    size=attn_mask.shape[-2:],
                    mode='nearest'
                )

                loss += 0.20 * F.mse_loss(
                    attn_mask,
                    t_attn
                )

        return loss


loss_fn = ComboDiceTverskyLoss(mode="full")

Training Cell

In [ ]:
# Training Cell
import os, gc, time, warnings
import pandas as pd
import torch
import numpy as np

warnings.filterwarnings("ignore")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =====================================================================
# Setup
# =====================================================================
os.makedirs("saved_models_AblationLoss", exist_ok=True)

EXPERIMENTS = [

    # 1. Simple Dice + BCE
    dict(
        name="Simple_DiceBCE",
        loss_mode="simple",
        use_25d=True,
        enc_att=False,
        dec_att=True,
        use_crop=False
    ),

    # 2. Hybrid loss
    dict(
        name="Hybrid",
        loss_mode="hybrid",
        use_25d=True,
        enc_att=False,
        dec_att=True,
        use_crop=False
    ),

    # 3. Hybrid + Auxiliary + Attention guidance
    dict(
        name="Hybrid_Aux_Attn",
        loss_mode="full",
        use_25d=True,
        enc_att=False,
        dec_att=True,
        use_crop=False
    ),

]

train_logs = []
results_df = []

print(f"Starting Loss Ablation Study with {len(EXPERIMENTS)} experiments...")

for i, cfg in enumerate(EXPERIMENTS):

    print(f"\n{'='*60}")
    print(f"Run {i+1}/{len(EXPERIMENTS)} : {cfg['name']}")
    print(f"{'='*60}")

    set_seed(SEED)

    current_train_ids = train_ids[:int(len(train_ids) * DATA_PERCENT)]
    current_val_ids   = val_ids[:int(len(val_ids) * DATA_PERCENT)]
    current_test_ids  = test_ids[:int(len(test_ids) * DATA_PERCENT)]

    dl_tr, dl_va, dl_te = make_loaders_ablation(
        current_train_ids,
        current_val_ids,
        current_test_ids,
        use_25d=cfg["use_25d"],
        use_crop=cfg["use_crop"]
    )

    in_channels = IN_CHANNELS if cfg["use_25d"] else 1

    model = SwinUNetAdaptive_DS(
        in_ch=in_channels,
        use_enc_att=cfg["enc_att"],
        use_dec_att=cfg["dec_att"]
    ).to(device)

    # Loss selected for this ablation
    loss_fn = ComboDiceTverskyLoss(
        mode=cfg["loss_mode"]
    ).to(device)

    # =================================================================
    # Optimizer (Differential LR)
    # =================================================================
    base_lr = 3e-4

    encoder_params = []
    attention_params = []
    decoder_and_head_params = []
    scale_params = []

    for name, param in model.named_parameters():

        if not param.requires_grad:
            continue

        if 'scale' in name:
            scale_params.append(param)

        elif any(
            key in name
            for key in [
                'enc_att',
                'dec_att',
                'local_attn',
                'global_attn',
                'gn'
            ]
        ):
            attention_params.append(param)

        elif 'encoder' in name:
            encoder_params.append(param)

        else:
            decoder_and_head_params.append(param)

    param_groups = [
        {
            'params': encoder_params,
            'lr': base_lr / 10
        },

        {
            'params': decoder_and_head_params,
            'lr': base_lr
        },

        {
            'params': attention_params,
            'lr': base_lr * 2
        },

        {
            'params': scale_params,
            'lr': 1e-3
        },
    ]

    optimizer = torch.optim.AdamW(
        param_groups,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=True
    )

    scaler = torch.cuda.amp.GradScaler()

    # =================================================================
    # Checkpointing based on Dice@0.5 only
    # =================================================================
    best_val_dice_05 = -1.0

    best_path = (
        f"saved_models_AblationLoss/"
        f"best_model_{cfg['name']}_S{SEED}.pt"
    )

    patience = 10
    patience_counter = 0
    start_time = time.time()

    for ep in range(1, EPOCHS + 1):

        tr_loss = train_one_epoch(
            model,
            dl_tr,
            optimizer,
            loss_fn,
            scaler,
            device
        )

        val_metrics_05 = eval_full_split(
            model,
            dl_va,
            loss_fn,
            device,
            calc_hd=False,
            thr=0.5
        )

        val_loss = val_metrics_05["loss"]
        val_dice_05 = val_metrics_05["dice"]

        scheduler.step(val_dice_05)

        if val_dice_05 > best_val_dice_05:

            best_val_dice_05 = val_dice_05

            torch.save(
                {
                    "epoch": ep,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_dice": best_val_dice_05,
                    "config": cfg,
                },
                best_path
            )

            patience_counter = 0

        else:
            patience_counter += 1

        train_logs.append({
            "Experiment": cfg["name"],
            "Epoch": ep,
            "LR": optimizer.param_groups[1]['lr'],
            "Train_Loss": tr_loss,
            "Val_Loss": val_loss,
            "Val_Dice@0.5": val_dice_05,
            "Best_Dice@0.5": best_val_dice_05,
        })

        if ep == 1 or ep % 5 == 0 or ep == EPOCHS:

            current_lr = optimizer.param_groups[1]['lr']

            print(
                f"   Ep {ep:03d}/{EPOCHS} | "
                f"LR: {current_lr:.2e} | "
                f"Train Loss: {tr_loss:.4f} | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val Dice@0.5: {val_dice_05:.4f} | "
                f"Best@0.5: {best_val_dice_05:.4f}"
            )

        if patience_counter >= patience:

            print(
                f"   Early Stopping Triggered! "
                f"No improvement for {patience} epochs."
            )

            break

    print(
        f"Training {cfg['name']} finished in "
        f"{(time.time() - start_time)/60:.1f} min."
    )

    # =================================================================
    # Load best checkpoint (selected by Dice@0.5)
    # =================================================================
    checkpoint = torch.load(
        best_path,
        map_location=device
    )

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

    elif isinstance(checkpoint, dict) and "model_state" in checkpoint:

        model.load_state_dict(
            checkpoint["model_state"]
        )

    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:

        model.load_state_dict(
            checkpoint["state_dict"]
        )

    else:

        model.load_state_dict(checkpoint)

    model.eval()

    print("Calculating final metrics (@0.5 only)...")

    with torch.no_grad():

        final_tr_05 = eval_full_split(
            model,
            dl_tr,
            loss_fn,
            device,
            calc_hd=False,
            thr=0.5
        )

        final_va_05 = eval_full_split(
            model,
            dl_va,
            loss_fn,
            device,
            calc_hd=True,
            thr=0.5
        )

        final_te_05 = eval_full_split(
            model,
            dl_te,
            loss_fn,
            device,
            calc_hd=True,
            thr=0.5
        )

    print(
        f"Final Dice Train/Val @0.5: "
        f"{final_tr_05.get('dice', 0):.4f} / "
        f"{final_va_05.get('dice', 0):.4f}"
    )

    # =================================================================
    # Save rows (Train + Val + Test at thr=0.5)
    # =================================================================
    for split_name, metrics_dict in [
        ("Train", final_tr_05),
        ("Val", final_va_05),
        ("Test", final_te_05)
    ]:

        row = cfg.copy()

        row["seed"] = SEED
        row["split"] = split_name
        row["eval_setting"] = "Dice@0.5"
        row["thr_used"] = 0.5

        row.update(metrics_dict)

        if "loss" in row:
            del row["loss"]

        row["best_val_dice@0.5"] = best_val_dice_05

        results_df.append(row)

    # =================================================================
    # Cleanup
    # =================================================================
    del model
    del optimizer
    del scheduler
    del scaler
    del loss_fn
    del dl_tr
    del dl_va
    del dl_te

    torch.cuda.empty_cache()
    gc.collect()


# =====================================================================
# Final table + save
# =====================================================================
print("\n" + "="*60)
print("ALL LOSS ABLATION EXPERIMENTS FINISHED. FINAL RESULTS:")
print("="*60)

df = pd.DataFrame(results_df)

cols = list(df.columns)

preferred_order = [
    'name',
    'use_25d',
    'enc_att',
    'dec_att',
    'use_crop',
    'seed',
    'split',
    'eval_setting',
    'thr_used',

    # main metrics
    'dice',
    'iou',
    'precision',
    'recall',
    'accuracy',
    'specificity',

    # spatial/boundary
    'hd95',
    'surface_dice',

    # complementary
    'f2_score',
    'mcc',
    'balanced_acc',
    'auc',
    'average_precision',

    # stats
    'recall_at_95_spec',
    'prevalence',
    'prediction_ratio',

    # confusion matrix
    'TP',
    'TN',
    'FP',
    'FN',

    # small/medium/large
    'dice_small',
    'dice_medium',
    'dice_large',
    'count_small',
    'count_medium',
    'count_large',

    # bookkeeping
    'best_val_dice@0.5'
]

final_cols = [
    c for c in preferred_order if c in cols
] + [
    c for c in cols if c not in preferred_order
]

df = df[final_cols]

csv_path = (
    f"Final_Ablation_Results_patch{PATCH_SIZE}_"
    f"{IMG_FORMAT}_S{SEED}.csv"
)

df.to_csv(
    csv_path,
    index=False
)

display(df)

print(f"Final results saved: {csv_path}")


# =====================================================================
# Training logs save
# =====================================================================
if train_logs:

    train_log_df = pd.DataFrame(train_logs)

    log_path = (
        f"Training_Logs_patch{PATCH_SIZE}_"
        f"{IMG_FORMAT}_S{SEED}.xlsx"
    )

    train_log_df.to_excel(
        log_path,
        index=False
    )

    print(f"Training logs saved: {log_path}")

----------------

In [5]:
# ============================================================
# LOSS ABLATION — VALIDATION RESULTS
# Article-Style Comparison Table
# Five independent runs (Seeds 41–45)
# Mean ± 95% Confidence Interval
# All numerical results are read directly from Excel.
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import t
from IPython.display import display


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEEDS = [41, 42, 43, 44, 45]

EXCEL_PATH = (
    Path("Results") /
    "Final_Loss_Ablation_Results_DecAtt_5Seeds.xlsx"
)

OUTPUT_DIR = Path("Results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. LOAD EXCEL DATA
# ============================================================

if not EXCEL_PATH.exists():
    raise FileNotFoundError(
        f"Excel file not found:\n{EXCEL_PATH.resolve()}"
    )

df = pd.read_excel(EXCEL_PATH)

# Normalize column names
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

required_columns = [
    "name",
    "seed",
    "split",
    "dice",
    "iou",
    "precision",
    "recall",
    "hd95",
    "empty_prediction_rate"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Excel columns: {missing_columns}\n"
        f"Available columns: {df.columns.tolist()}"
    )


# ============================================================
# 3. FILTER VALIDATION RESULTS ONLY
# ============================================================

df["split"] = df["split"].astype(str).str.strip().str.lower()

df_val = df[
    df["split"].isin(["val", "valid", "validation"])
].copy()

df_val["seed"] = pd.to_numeric(
    df_val["seed"], errors="coerce"
)

df_val = df_val[
    df_val["seed"].isin(SEEDS)
].copy()

df_val["name"] = df_val["name"].astype(str).str.strip()

EXPERIMENTS = [
    "Simple_DiceBCE",
    "Hybrid",
    "Hybrid_Aux_Att"
]

# Keep only the three loss ablation experiments
df_val = df_val[
    df_val["name"].isin(EXPERIMENTS)
].copy()

if df_val.empty:
    raise ValueError(
        "No validation records were found for the three experiments."
    )


# ============================================================
# 4. CHECK DATA COMPLETENESS
# ============================================================

for experiment in EXPERIMENTS:

    exp_df = df_val[df_val["name"] == experiment]

    found_seeds = sorted(
        exp_df["seed"].dropna().astype(int).unique().tolist()
    )

    if found_seeds != SEEDS:
        raise ValueError(
            f"{experiment}: Expected seeds {SEEDS}, "
            f"but found {found_seeds}."
        )

    if exp_df["seed"].duplicated().any():
        raise ValueError(
            f"{experiment}: More than one validation row "
            "was found for at least one seed."
        )

    if len(exp_df) != 5:
        raise ValueError(
            f"{experiment}: Expected 5 validation rows, "
            f"but found {len(exp_df)}."
        )

print("Validation data loaded successfully.")
print(f"Total validation records: {len(df_val)}")
print(f"Experiments: {EXPERIMENTS}")
print(f"Seeds: {SEEDS}")


# ============================================================
# 5. CONVERT METRICS TO PERCENTAGES WHERE NECESSARY
# ============================================================

percentage_columns = [
    "dice",
    "iou",
    "precision",
    "recall",
    "empty_prediction_rate"
]

for col in percentage_columns:

    df_val[col] = pd.to_numeric(
        df_val[col], errors="coerce"
    )

    # Convert proportions (0–1) to percentages (0–100)
    if df_val[col].max() <= 1:
        df_val[col] = df_val[col] * 100


df_val["hd95"] = pd.to_numeric(
    df_val["hd95"], errors="coerce"
)


# Check that all required numerical values are available
metric_columns = [
    "dice",
    "iou",
    "precision",
    "recall",
    "hd95",
    "empty_prediction_rate"
]

if df_val[metric_columns].isna().any().any():
    missing = df_val[
        df_val[metric_columns].isna().any(axis=1)
    ][["name", "seed"] + metric_columns]

    raise ValueError(
        "Missing numerical values were found in the validation data:\n"
        f"{missing.to_string(index=False)}"
    )


# ============================================================
# 6. CALCULATE MEAN ± 95% CI
# ============================================================

def calculate_mean_ci(values):

    values = np.asarray(values, dtype=float)

    n = len(values)

    if n != 5:
        raise ValueError(
            f"Expected 5 values, but received {n}."
        )

    mean = np.mean(values)
    sd = np.std(values, ddof=1)

    ci = t.ppf(0.975, df=n - 1) * sd / np.sqrt(n)

    return mean, ci


def format_percentage(values):

    mean, ci = calculate_mean_ci(values)

    return f"{mean:.2f} ± {ci:.2f}%"


def format_hd95(values):

    mean, ci = calculate_mean_ci(values)

    return f"{mean:.2f} ± {ci:.2f}"


# ============================================================
# 7. CREATE ARTICLE-STYLE COMPARISON TABLE
# ============================================================

experiment_labels = {
    "Simple_DiceBCE": "Simple_DiceBCE",
    "Hybrid": "Hybrid",
    "Hybrid_Aux_Att": (
        "Hybrid_Aux_Att (final 2.5D_DecAtt model)"
    )
}

table_rows = []


def add_row(metric, values):

    table_rows.append({
        "Experiments": metric,
        **values
    })


# ------------------------------------------------------------
# Loss function components
# ------------------------------------------------------------

add_row("Dice loss", {
    "Simple_DiceBCE": "✓",
    "Hybrid": "✓",
    "Hybrid_Aux_Att": "✓"
})

add_row("BCE loss", {
    "Simple_DiceBCE": "✓",
    "Hybrid": "✓",
    "Hybrid_Aux_Att": "✓"
})

add_row("Focal Tversky loss", {
    "Simple_DiceBCE": "−",
    "Hybrid": "✓",
    "Hybrid_Aux_Att": "✓"
})

add_row("Boundary loss", {
    "Simple_DiceBCE": "−",
    "Hybrid": "✓",
    "Hybrid_Aux_Att": "✓"
})

add_row("Auxiliary BCE loss", {
    "Simple_DiceBCE": "−",
    "Hybrid": "−",
    "Hybrid_Aux_Att": "✓"
})

add_row("Attention guidance loss", {
    "Simple_DiceBCE": "−",
    "Hybrid": "−",
    "Hybrid_Aux_Att": "✓"
})


# ------------------------------------------------------------
# Performance metrics — Validation set
# ------------------------------------------------------------

metric_mapping = [
    ("Dice score", "dice", format_percentage),
    ("IoU", "iou", format_percentage),
    ("Sensitivity", "recall", format_percentage),
    ("Precision", "precision", format_percentage),
    ("HD95 (px)", "hd95", format_hd95),
    (
        "Empty prediction rate",
        "empty_prediction_rate",
        format_percentage
    )
]

for display_name, column_name, formatter in metric_mapping:

    values = {}

    for experiment in EXPERIMENTS:

        exp_df = df_val[
            df_val["name"] == experiment
        ].sort_values("seed")

        values[experiment] = formatter(
            exp_df[column_name].to_numpy()
        )

    add_row(display_name, values)


# Final article-style table
article_table = pd.DataFrame(table_rows)

# Rename the third experiment column for presentation
article_table = article_table.rename(
    columns={
        "Hybrid_Aux_Att":
        "Hybrid_Aux_Att (final 2.5D_DecAtt model)"
    }
)

print("\n" + "=" * 110)
print("LOSS ABLATION — VALIDATION RESULTS")
print("Mean ± 95% CI across five independent runs")
print("=" * 110)

display(article_table)


# ============================================================
# 8. PER-SEED VALIDATION RESULTS
# ============================================================

per_seed_records = []

for experiment in EXPERIMENTS:

    exp_df = df_val[
        df_val["name"] == experiment
    ].sort_values("seed")

    for _, row in exp_df.iterrows():

        per_seed_records.append({
            "Experiment": experiment_labels[experiment],
            "Seed": int(row["seed"]),
            "Dice (%)": row["dice"],
            "IoU (%)": row["iou"],
            "Sensitivity (%)": row["recall"],
            "Precision (%)": row["precision"],
            "HD95 (px)": row["hd95"],
            "Empty prediction rate (%)":
                row["empty_prediction_rate"]
        })

per_seed_df = pd.DataFrame(per_seed_records)

print("\n" + "=" * 110)
print("PER-SEED VALIDATION RESULTS")
print("=" * 110)

display(
    per_seed_df.style.format({
        "Dice (%)": "{:.2f}",
        "IoU (%)": "{:.2f}",
        "Sensitivity (%)": "{:.2f}",
        "Precision (%)": "{:.2f}",
        "HD95 (px)": "{:.4f}",
        "Empty prediction rate (%)": "{:.2f}"
    })
)


# ============================================================
# 9. EXPORT RESULTS
# ============================================================

article_table.to_excel(
    OUTPUT_DIR / "Loss_Ablation_Article_Table.xlsx",
    index=False
)

per_seed_df.to_excel(
    OUTPUT_DIR / "Loss_Ablation_PerSeed_Validation.xlsx",
    index=False
)

print("\n" + "=" * 110)
print("FILES EXPORTED SUCCESSFULLY")
print("=" * 110)

print(
    OUTPUT_DIR / "Loss_Ablation_Article_Table.xlsx"
)

print(
    OUTPUT_DIR / "Loss_Ablation_PerSeed_Validation.xlsx"
)

Validation data loaded successfully.
Total validation records: 15
Experiments: ['Simple_DiceBCE', 'Hybrid', 'Hybrid_Aux_Att']
Seeds: [41, 42, 43, 44, 45]

LOSS ABLATION — VALIDATION RESULTS
Mean ± 95% CI across five independent runs


,Experiments,Simple_DiceBCE,Hybrid,Hybrid_Aux_Att (final 2.5D_DecAtt model)
0,Dice loss,✓,✓,✓
1,BCE loss,✓,✓,✓
2,Focal Tversky loss,−,✓,✓
3,Boundary loss,−,✓,✓
4,Auxiliary BCE loss,−,−,✓
5,Attention guidance loss,−,−,✓
6,Dice score,72.05 ± 1.40%,72.87 ± 2.25%,74.88 ± 2.18%
7,IoU,56.32 ± 1.71%,57.35 ± 2.77%,59.87 ± 2.79%
8,Sensitivity,65.67 ± 3.02%,67.75 ± 3.22%,69.99 ± 3.69%
9,Precision,79.89 ± 2.18%,78.89 ± 2.23%,80.57 ± 1.00%



PER-SEED VALIDATION RESULTS


,Experiment,Seed,Dice (%),IoU (%),Sensitivity (%),Precision (%),HD95 (px),Empty prediction rate (%)
0,Simple_DiceBCE,41,73.20,57.72,66.65,81.17,23.6257,14.31
1,Simple_DiceBCE,42,70.47,54.40,61.74,82.07,19.2792,16.47
2,Simple_DiceBCE,43,72.22,56.52,67.54,77.59,27.8878,13.35
3,Simple_DiceBCE,44,72.96,57.43,67.46,79.44,21.4683,16.38
4,Simple_DiceBCE,45,71.39,55.50,64.98,79.20,22.9604,15.71
5,Hybrid,41,73.54,58.15,68.83,78.95,21.2960,14.17
6,Hybrid,42,72.81,57.25,66.81,80.00,18.6707,18.16
7,Hybrid,43,70.17,54.05,63.66,78.15,22.9951,16.67
8,Hybrid,44,75.18,60.23,70.12,81.02,15.4034,14.31
9,Hybrid,45,72.66,57.06,69.33,76.32,24.5268,10.95



FILES EXPORTED SUCCESSFULLY
Results\Loss_Ablation_Article_Table.xlsx
Results\Loss_Ablation_PerSeed_Validation.xlsx
